# 12. SQL Window Functions: Over, Rank & Running Totals: Beginner Guide

### 📝 Universal SQL Execution Order (All Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline ─────────────────────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. SELECT & CASE   ➔ 8. DISTINCT (Dedup)   ➔ 9. ORDER BY (Sorting)         │
│ ➔ 10. LIMIT / OFFSET (Final Page Slice)                                      │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **12. SQL Window Functions: Over, Rank & Running Totals**. Unlike standard `GROUP BY` aggregations that collapse rows into summary tuples, window functions compute metric calculations across related sliding row frames while *preserving the individual identity of every row*. This notebook covers the `OVER()` clause, window partitioning (`PARTITION BY`), deterministic ordering (`ORDER BY`), ranking functions (`ROW_NUMBER`, `RANK`, `DENSE_RANK`), inter-row relative access (`LAG`, `LEAD`), running totals, and moving averages via `ROWS BETWEEN`.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Window Anatomy & Frame Specification: `OVER (PARTITION BY ... ORDER BY ...)`
- [x] 🔹 Partition Isolation: Resetting Metric Scopes across Categories
- [x] 🔹 Deterministic Ranking: `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`
- [x] 🔹 Adjacent Row Inspection: `LAG(col, offset)` & `LEAD(col, offset)`
- [x] 🔹 Sliding Bounded Frames: `ROWS BETWEEN N PRECEDING AND CURRENT ROW`
- [x] 🔍 Scenario: Calculating Customer Inter-Purchase Velocities & Running Balances










In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Sequence Ranking: `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`
- **What it does:** Assigns integer ranks to rows within partitions ordered by designated sort keys.
- **Syntax:** `RANK() OVER (PARTITION BY group_col ORDER BY metric_col DESC)`
- **Dataset Application & Code Demonstration:** Ranks the largest transactions within each region.


In [2]:
%%sql
SELECT 
    region,
    transaction_id,
    customer_id,
    transaction_amount,
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY transaction_amount DESC) AS row_num,
    DENSE_RANK() OVER (PARTITION BY region ORDER BY transaction_amount DESC) AS dense_rnk
FROM transactions
WHERE transaction_amount IS NOT NULL
LIMIT 8;


,region,transaction_id,customer_id,transaction_amount,row_num,dense_rnk
0,East,TX112849,C42493,1913.72,1,1
1,East,TX110923,C25194,1882.23,2,2
2,East,TX112042,C60140,1871.15,3,3
3,East,TX109920,C12540,1835.03,4,4
4,East,TX112710,C50492,1823.22,5,5
5,East,TX108316,C62386,1749.71,6,6
6,East,TX108255,C70443,1722.70,7,7
7,East,TX113853,C81200,1720.52,8,8


### 🔹 Relative Inter-Row Inspection: `LAG()` & `LEAD()`
- **What it does:** Accesses column values from preceding (`LAG`) or succeeding (`LEAD`) rows in the ordered window without self-joins.
- **Syntax:** `LAG(column_name, offset, default_val) OVER (PARTITION BY ... ORDER BY ...)`
- **Dataset Application & Code Demonstration:** Calculates delta between a transaction and the customer's previous transaction.


In [3]:
%%sql
SELECT 
    customer_id,
    transaction_id,
    transaction_amount,
    LAG(transaction_amount, 1, 0.0) OVER (PARTITION BY customer_id ORDER BY transaction_id ASC) AS prev_amount,
    ROUND(transaction_amount - LAG(transaction_amount, 1, transaction_amount) OVER (PARTITION BY customer_id ORDER BY transaction_id ASC), 2) AS spend_delta
FROM transactions
WHERE transaction_amount IS NOT NULL
LIMIT 6;


,customer_id,transaction_id,transaction_amount,prev_amount,spend_delta
0,C10053,TX102041,1908.10,0.00,0.00
1,C10053,TX103910,719.86,1908.10,-1188.24
2,C10053,TX105377,645.33,719.86,-74.53
3,C10053,TX105459,1605.63,645.33,960.30
4,C10053,TX107036,1764.55,1605.63,158.92
5,C10053,TX107255,404.98,1764.55,-1359.57


### 🔹 Bounded Sliding Frames: Running Cumulative Totals
- **What it does:** Computes rolling aggregates (sum, average) over explicit physical row frames.
- **Syntax:** `SUM(col) OVER (PARTITION BY grp ORDER BY sort_key ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`
- **Dataset Application & Code Demonstration:** Computes cumulative customer spend progression.


In [4]:
%%sql
SELECT 
    customer_id,
    transaction_id,
    transaction_amount,
    ROUND(SUM(transaction_amount) OVER (
        PARTITION BY customer_id 
        ORDER BY transaction_id ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS running_total_spend
FROM transactions
WHERE transaction_amount IS NOT NULL
LIMIT 6;


,customer_id,transaction_id,transaction_amount,running_total_spend
0,C10053,TX102041,1908.10,1908.10
1,C10053,TX103910,719.86,2627.96
2,C10053,TX105377,645.33,3273.29
3,C10053,TX105459,1605.63,4878.92
4,C10053,TX107036,1764.55,6643.47
5,C10053,TX107255,404.98,7048.45


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Top 2 Highest Transactions per Region Filtered via CTE
- **Objective:** Extract the top 2 highest-value transactions in each region.
- **Approach:** Compute `DENSE_RANK() OVER (PARTITION BY region ...)` in a CTE and filter `WHERE rnk <= 2` in the outer query.


In [5]:
%%sql
WITH ranked_transactions AS (
    SELECT 
        region,
        transaction_id,
        customer_id,
        transaction_amount,
        DENSE_RANK() OVER (PARTITION BY region ORDER BY transaction_amount DESC) AS rnk
    FROM transactions
    WHERE region IS NOT NULL AND transaction_amount IS NOT NULL
)
SELECT region, transaction_id, customer_id, transaction_amount, rnk
FROM ranked_transactions
WHERE rnk <= 2
ORDER BY region ASC, rnk ASC;


,region,transaction_id,customer_id,transaction_amount,rnk
0,East,TX112849,C42493,1913.72,1
1,East,TX110923,C25194,1882.23,2
2,North,TX101298,C29080,1969.61,1
3,North,TX109032,C43408,1938.18,2
4,South,TX114444,C59941,1956.59,1
5,South,TX103634,C34356,1939.96,2
6,West,TX113712,C89840,1994.40,1
7,West,TX101474,C88139,1962.62,2
8,East,TX113287,C64524,1999.41,1
9,East,TX109638,C12260,1999.18,2
